# Public practice input equivalence gate

CPU only; validates public PyPI artifacts, 25 practice versions and exact seeded RESET observations against prior direct public play. No competition datasource, entry or submission.

In [ ]:
from pathlib import Path
from hashlib import sha256
import importlib.metadata
import json,subprocess,sys
root=Path('/kaggle/input/datasets/kunalsinghh03/arc3-public-development-inputs')
if not root.is_dir():root=Path('/kaggle/input/arc3-public-development-inputs')
assert root.is_dir(), 'Public development dataset not mounted'
print('PUBLIC INPUT ROOT',str(root),flush=True)
print('PUBLIC INPUT CONTENTS',sorted(p.name for p in root.iterdir()),flush=True)
manifest=json.loads((root/'public-input-manifest.json').read_text())
for item in manifest['environment_files']:
    assert sha256((root/item['file']).read_bytes()).hexdigest()==item['sha256']
for item in manifest['pypi_wheels']:
    assert sha256((root/'arc_agi_3_wheels'/item['file']).read_bytes()).hexdigest()==item['sha256']
subprocess.run([sys.executable,'-m','pip','install','--quiet','--no-index','--no-warn-conflicts',
               '--find-links',str(root/'arc_agi_3_wheels'),'arc-agi==0.9.8','arcengine==0.9.3'],check=True)
import numpy as np
from arc_agi import Arcade,OperationMode
arc=Arcade(operation_mode=OperationMode.OFFLINE,environments_dir=str(root/'environment_files'))
ids=sorted(e.game_id for e in arc.get_environments())
assert ids==manifest['game_ids'], 'Public game set differs'
expected={'ar25-0c556536': '537bdfbc5b53b0e0b1c0e3d25884bad3d25844934f93ddaa8921807a49330df5', 'bp35-0a0ad940': '92370b9db1c1b1f1570ec24487c8a1669fe3bf35572f0c3efa2bb084c5a1a634', 'cd82-fb555c5d': '55fc86753adfa398ca49054622171804e0fccb169d6f2cff005a5da3380c29cf', 'cn04-2fe56bfb': '2f49bfb6ed8ecaa5fdbcee7c7aca2a3b4ac66185cb4e79e6fc69597c30df7f6e', 'dc22-fdcac232': 'e2551bca84fde5ff7bfcf27cffd42b51e35eeaf9461a6e59e9658fe2dbdd5397', 'ft09-0d8bbf25': '79fe55ca830e50d246324fb6441268130cd6302b670f7365e4da995d102a5b1a', 'g50t-5849a774': '8b1eb3dd6466483f42ed9ba0c306627f97ae99e7f2eb566352b8b06bea1c51ac', 'ka59-38d34dbb': 'faf6d467c6e91fb9828c15f067a534d1d210175b77c8827ab4bc25926a599029', 'lf52-271a04aa': '9ce3975d61cdd3dce8a2e0215a8c50dbbcf248cc467f061439a1ae8b5b439d37', 'lp85-305b61c3': '06ca3fed44d537145688eb6e38a6fee5de66ff9be09ec1f548e5527f963dfd29', 'ls20-9607627b': '6305f5acb31a6c46bd260d54eb9c29dcddc91803a21189cd672ae4c0778b3f17', 'm0r0-492f87ba': 'bf50ade2766df302e6c83a2bcde43996d62900652bfa81f3577f6c721f4b65c1', 'r11l-495a7899': '8b85f85e19e9652a92a47a6eb26d3406b55cb8ae00057f76782b74c30ee5289c', 're86-8af5384d': '5f0de7b774cc57313626598acb4d331ccc02a120aaa63a0f148c9c70730e8348', 's5i5-18d95033': '3a5876c2868f410844fed4d32085fe36a466ea6d66174e0cd6d2d3b13016ea62', 'sb26-7fbdac44': '92b851143bc2102c68f0421aa7b74b1933fca5fca7505a99b00293f95d3ab03f', 'sc25-635fd71a': '50be52240020cd5e8d3d6944e98bd3efb57d6d6614bf15a950df18d53c8eb9c4', 'sk48-d8078629': '7c3785d89151d700965bdafc7560f6cd2d125df6ba20b0f092aa46f16a1b52f9', 'sp80-589a99af': '4537a63ec78645d0b682cfc1bc96b1d1dfd54acb52276e0451bf1a2358084198', 'su15-1944f8ab': '4f7a9e9c9162cd2264988122c927256ad18504b1c6bc4407c8ff248e96483686', 'tn36-ef4dde99': 'b68903aa844a182399a43b6c45b4168a43fc58fbcbb13a62d728e26777f537fd', 'tr87-cd924810': 'b9139ae3c22999de25e673cea24177afeb012cf49f2f3525b5eb1137a6193413', 'tu93-0768757b': '8ef2bbf7166c13a69b9fac884bb016f757fb44332a74fd530bae8b315d7f9535', 'vc33-5430563c': 'fd97797a15b28f39ed2589ad62573d4c53fa6d17270d693b4ac765a7ec8811d6', 'wa30-ee6fef47': '86eded3324f15ce897a3dfed1ab9317bb9f282cbc3448818641740043c59fb82'}
checks=[]
for game_id in ids:
    env=arc.make(game_id,seed=0)
    observation=env.reset()
    frame=np.asarray(observation.frame[-1],dtype=np.int16)
    digest=sha256(frame.tobytes()).hexdigest()
    assert digest==expected[game_id], 'Initial public observation differs: '+game_id
    checks.append(dict(game_id=game_id,shape=list(frame.shape),initial_observation_sha256=digest,total_levels=observation.win_levels))
report=dict(passed=True,games=len(checks),total_levels=sum(c['total_levels'] for c in checks),
            versions={name:importlib.metadata.version(name) for name in ('arc-agi','arcengine')},checks=checks,
            boundary='Offline public input/hash/RESET equivalence only. No model gameplay or score measured.')
Path('/kaggle/working/public-input-probe.json').write_text(json.dumps(report,indent=2))
print('PUBLIC_INPUT_CPU_PROBE_PASSED',json.dumps(report),flush=True)
